# NFL Fourth Down Decision Predictor — Joshua McCann

Run every cell top to bottom (**Runtime → Run all**). The first cells write out the project's `.py` files, then install packages, then train the models against the real nflverse data (needs internet — Colab has it), then get a sample prediction.

When it finishes, `results/summary.md` has the numbers for your report, and `results/figures/` has the two charts.

## 1. Write out the project files

In [ ]:
%%writefile config.py
"""Shared settings for the NFL fourth-down decision project."""
from pathlib import Path

ROOT = Path(__file__).resolve().parent
DATA_DIR = ROOT / "data" / "raw"            # downloaded nflverse files are cached here
PROCESSED_DIR = ROOT / "data" / "processed" # cleaned fourth-down table
MODEL_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"

# nflverse play-by-play release files (public, no login needed)
BASE_URL = "https://github.com/nflverse/nflverse-data/releases/download/pbp"

# Seasons to use. The LAST season is held out as the test set (time-based split),
# so the model is always evaluated on games it has never seen.
FIRST_SEASON = 2019
LAST_SEASON = 2024

# Model inputs, taken from the approved proposal:
#   field position, yards to go, quarter, time left, score difference, home/away
FEATURES = [
    "yardline_100",            # yards from the opponent's end zone (35 = opponent's 35)
    "ydstogo",                 # yards needed for a first down
    "qtr",                     # 1-4, 5 = overtime
    "game_seconds_remaining",  # seconds left in the game
    "score_differential",      # offense score minus defense score
    "is_home",                 # 1 if the offense is the home team
]
TARGET = "go_for_it"           # 1 = went for it (run/pass), 0 = kicked (punt/field goal)

# Simple non-AI baseline: "go for it on short yardage when in the opponent's half"
RULE_MAX_YDSTOGO = 2
RULE_MAX_YARDLINE_100 = 50

RANDOM_STATE = 42


In [ ]:
%%writefile baseline.py
"""Non-AI baselines the machine learning models must beat."""
import pandas as pd
from sklearn.metrics import f1_score

import config


def always_kick(df: pd.DataFrame):
    """Majority-class baseline: never go for it."""
    return pd.Series(0, index=df.index)


def rule_based_go(df: pd.DataFrame,
                  max_ydstogo: int = config.RULE_MAX_YDSTOGO,
                  max_yardline_100: int = config.RULE_MAX_YARDLINE_100):
    """Simple rule: go for it on short yardage when in the opponent's half."""
    return ((df["ydstogo"] <= max_ydstogo) &
            (df["yardline_100"] <= max_yardline_100)).astype(int)


def tune_rule(train_df: pd.DataFrame):
    """Search the best (yards-to-go, field-position) cutoffs on TRAINING data only.

    This gives the ML models a stronger, fairer opponent than a hand-picked rule.
    """
    best = (-1.0, config.RULE_MAX_YDSTOGO, config.RULE_MAX_YARDLINE_100)
    for ytg in range(1, 11):
        for yl in range(20, 101, 5):
            pred = rule_based_go(train_df, ytg, yl)
            f1 = f1_score(train_df[config.TARGET], pred, zero_division=0)
            if f1 > best[0]:
                best = (f1, ytg, yl)
    return {"max_ydstogo": best[1], "max_yardline_100": best[2], "train_f1": best[0]}


In [ ]:
%%writefile data.py
"""Download, load, and clean nflverse play-by-play data for fourth-down decisions."""
import shutil
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

import config

try:  # parquet is smaller/faster, but needs pyarrow
    import pyarrow.parquet as pq
    HAS_PYARROW = True
except ImportError:
    HAS_PYARROW = False

# Only these columns are kept from the ~370 in the raw file.
NEEDED_COLS = [
    "season", "season_type", "game_id", "play_id", "posteam", "home_team",
    "posteam_type", "down", "ydstogo", "yardline_100", "qtr",
    "game_seconds_remaining", "score_differential", "play_type",
    "fourth_down_converted", "fourth_down_failed",
]
REQUIRED_COLS = ["season", "season_type", "game_id", "down", "ydstogo",
                 "yardline_100", "qtr", "game_seconds_remaining",
                 "score_differential", "play_type"]


def _download(url: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")
    req = urllib.request.Request(url, headers={"User-Agent": "fourth-down-student-project"})
    with urllib.request.urlopen(req, timeout=180) as resp, open(tmp, "wb") as f:
        shutil.copyfileobj(resp, f)
    tmp.replace(dest)


def _read_parquet(path: Path) -> pd.DataFrame:
    cols = [c for c in NEEDED_COLS if c in pq.ParquetFile(path).schema.names]
    return pd.read_parquet(path, columns=cols)


def _read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, usecols=lambda c: c in NEEDED_COLS, low_memory=False)


def load_season(season: int, data_dir: Path = config.DATA_DIR) -> pd.DataFrame:
    """Load one season of play-by-play, downloading it the first time."""
    pq_path = data_dir / f"play_by_play_{season}.parquet"
    csv_path = data_dir / f"play_by_play_{season}.csv.gz"

    if HAS_PYARROW and pq_path.exists():
        return _read_parquet(pq_path)
    if csv_path.exists():
        return _read_csv(csv_path)

    if HAS_PYARROW:
        try:
            print(f"  downloading {season} (parquet)...")
            _download(f"{config.BASE_URL}/play_by_play_{season}.parquet", pq_path)
            return _read_parquet(pq_path)
        except Exception as err:  # fall back to CSV
            print(f"  parquet failed ({err}); trying csv.gz")
    print(f"  downloading {season} (csv.gz)...")
    _download(f"{config.BASE_URL}/play_by_play_{season}.csv.gz", csv_path)
    return _read_csv(csv_path)


def load_seasons(seasons, data_dir: Path = config.DATA_DIR) -> pd.DataFrame:
    frames = []
    for s in seasons:
        print(f"Loading {s}")
        frames.append(load_season(s, data_dir))
    pbp = pd.concat(frames, ignore_index=True)
    missing = [c for c in REQUIRED_COLS if c not in pbp.columns]
    if missing:
        raise KeyError(f"Expected columns missing from the data: {missing}")
    return pbp


def clean_fourth_downs(pbp: pd.DataFrame):
    """Keep regular-season 4th-down decisions and build the label.

    go_for_it = 1 when the offense ran or passed (an actual attempt)
    go_for_it = 0 when it punted or kicked a field goal
    QB kneels/spikes and penalty 'no_play' rows are dropped (not real decisions).
    """
    counts = {"raw_plays": int(len(pbp))}

    df = pbp[(pbp["season_type"] == "REG") & (pbp["down"] == 4)].copy()
    counts["regular_season_fourth_downs"] = int(len(df))

    df = df[df["play_type"].isin(["pass", "run", "punt", "field_goal"])].copy()
    counts["fourth_down_decisions"] = int(len(df))

    df[config.TARGET] = df["play_type"].isin(["pass", "run"]).astype(int)

    if "posteam_type" in df.columns:
        df["is_home"] = (df["posteam_type"] == "home").astype(int)
    else:
        df["is_home"] = (df["posteam"] == df["home_team"]).astype(int)

    # Conversion result (only meaningful for go-for-it plays); used by the backup model.
    conv = df["fourth_down_converted"] if "fourth_down_converted" in df.columns else np.nan
    df["converted"] = np.where(df[config.TARGET] == 1, conv, np.nan)

    before = len(df)
    df = df.dropna(subset=config.FEATURES + ["season"])
    counts["dropped_missing_values"] = int(before - len(df))
    counts["rows_kept"] = int(len(df))
    counts["go_for_it_rate"] = round(float(df[config.TARGET].mean()), 4)

    keep = ["season", "game_id", "play_id", "posteam", "play_type"] + config.FEATURES + \
           [config.TARGET, "converted"]
    df = df[[c for c in keep if c in df.columns]].reset_index(drop=True)
    return df, counts


def save_processed(df: pd.DataFrame) -> Path:
    config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    path = config.PROCESSED_DIR / "fourth_down_decisions.csv"
    df.to_csv(path, index=False)
    df.head(25).to_csv(config.PROCESSED_DIR / "sample_25_rows.csv", index=False)
    return path


In [ ]:
%%writefile train.py
"""Train and evaluate the fourth-down decision models.

Usage:
    python train.py                       # seasons in config.py (downloads data first time)
    python train.py --first-season 2018 --last-season 2024

Outputs (all under results/ and models/):
    results/metrics.json          every number used in the report
    results/summary.md            ready-to-copy tables for the checkpoint template
    results/figures/*.png         figures to paste into the report
    models/best_model.joblib      go-for-it model chosen by cross-validation
    models/conversion_model.joblib  backup model: P(convert | go for it)
"""
import argparse
import json
import textwrap

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import config
from baseline import always_kick, rule_based_go, tune_rule
from data import clean_fourth_downs, load_seasons, save_processed


def evaluate(y_true, y_pred, y_prob=None):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob) if y_prob is not None else None,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }
    return {k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v)
            for k, v in out.items()}


def make_models():
    return {
        "Logistic Regression": make_pipeline(
            StandardScaler(),
            LogisticRegression(class_weight="balanced", max_iter=1000,
                               random_state=config.RANDOM_STATE)),
        "Random Forest": RandomForestClassifier(
            n_estimators=300, min_samples_leaf=5, class_weight="balanced",
            random_state=config.RANDOM_STATE, n_jobs=-1),
    }


def plot_go_rate(df):
    d = df.assign(ytg=df["ydstogo"].clip(upper=15))
    rate = d.groupby("ytg")[config.TARGET].mean()
    fig, ax = plt.subplots(figsize=(6.5, 3.4))
    ax.bar(rate.index.astype(int), rate.values * 100, color="#2b6cb0")
    ax.set_xlabel("Yards to go on 4th down (15 = 15+)")
    ax.set_ylabel("Teams that went for it (%)")
    ax.set_title("Historical 4th-down go-for-it rate by distance")
    fig.tight_layout()
    path = config.FIGURES_DIR / "go_rate_by_yards_to_go.png"
    fig.savefig(path, dpi=200)
    plt.close(fig)
    return path


def plot_comparison(results):
    names = list(results)
    x = np.arange(len(names))
    fig, ax = plt.subplots(figsize=(7.5, 3.6))
    ax.bar(x - 0.2, [results[n]["accuracy"] for n in names], 0.4, label="Accuracy", color="#a0aec0")
    ax.bar(x + 0.2, [results[n]["f1"] for n in names], 0.4, label="F1 (go for it)", color="#2b6cb0")
    ax.set_xticks(x)
    ax.set_xticklabels([textwrap.fill(n, 16) for n in names], fontsize=7)
    ax.set_ylim(0, 1)
    ax.set_title("Held-out test season: baselines vs. machine learning")
    ax.legend(loc="upper left", fontsize=8)
    fig.tight_layout()
    path = config.FIGURES_DIR / "model_comparison.png"
    fig.savefig(path, dpi=200)
    plt.close(fig)
    return path


def fmt(v):
    return "n/a" if v is None else f"{v:.3f}"


def write_summary(counts, split, tuned, results, cv, conv, importances):
    lines = ["# Results summary (copy these numbers into the checkpoint report)", ""]
    lines += ["## Data",
              f"- Seasons loaded: {split['seasons']}",
              f"- Raw plays loaded: {counts['raw_plays']:,}",
              f"- Regular-season 4th-down plays: {counts['regular_season_fourth_downs']:,}",
              f"- 4th-down decisions kept (go / punt / field goal): {counts['rows_kept']:,}",
              f"- Go-for-it rate: {counts['go_for_it_rate'] * 100:.1f}%",
              f"- Train: {split['train_n']:,} plays (seasons {split['train_seasons']}); "
              f"Test: {split['test_n']:,} plays (season {split['test_season']})", ""]
    lines += [f"## Test-season results (season {split['test_season']})",
              "| Method | Accuracy | Precision | Recall | F1 (go) | ROC AUC |",
              "|---|---|---|---|---|---|"]
    for name, r in results.items():
        lines.append(f"| {name} | {fmt(r['accuracy'])} | {fmt(r['precision'])} | "
                     f"{fmt(r['recall'])} | {fmt(r['f1'])} | {fmt(r['roc_auc'])} |")
    lines += ["", "## Cross-validation on training seasons (F1, grouped by game)"]
    for name, s in cv.items():
        lines.append(f"- {name}: {s['mean']:.3f} +/- {s['std']:.3f}")
    lines += ["", f"Best model by cross-validation: **{split['best_model']}**", ""]
    lines += ["## Backup model: conversion success (go-for-it plays only)",
              f"- Test attempts: {conv['test_n']}; actual conversion rate {conv['test_rate']:.3f}",
              f"- ROC AUC {conv['roc_auc']:.3f}; accuracy {conv['accuracy']:.3f}", ""]
    lines += ["## Random Forest feature importance"]
    for k, v in sorted(importances.items(), key=lambda kv: -kv[1]):
        lines.append(f"- {k}: {v:.3f}")
    (config.RESULTS_DIR / "summary.md").write_text("\n".join(lines) + "\n")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--first-season", type=int, default=config.FIRST_SEASON)
    ap.add_argument("--last-season", type=int, default=config.LAST_SEASON)
    args = ap.parse_args()

    config.RESULTS_DIR.mkdir(exist_ok=True)
    config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    config.MODEL_DIR.mkdir(exist_ok=True)

    # ---- data ----
    seasons = list(range(args.first_season, args.last_season + 1))
    pbp = load_seasons(seasons)
    df, counts = clean_fourth_downs(pbp)
    save_processed(df)
    print("\nData counts:", json.dumps(counts, indent=2))

    test_season = args.last_season
    train_df = df[df["season"] < test_season]
    test_df = df[df["season"] == test_season]
    if train_df.empty or test_df.empty:
        raise SystemExit("Need at least two seasons (one to train on, one to test on).")
    X_tr, y_tr = train_df[config.FEATURES], train_df[config.TARGET]
    X_te, y_te = test_df[config.FEATURES], test_df[config.TARGET]
    split = {"seasons": f"{seasons[0]}-{seasons[-1]}", "train_seasons": f"{seasons[0]}-{test_season - 1}",
             "test_season": test_season, "train_n": len(train_df), "test_n": len(test_df)}

    # ---- baselines ----
    tuned = tune_rule(train_df)
    hand_name = f"Rule: 4th & <={config.RULE_MAX_YDSTOGO}, opp. half"
    tuned_name = f"Tuned rule: 4th & <={tuned['max_ydstogo']}, opp. {tuned['max_yardline_100']}"
    results = {
        "Always kick (majority)": evaluate(y_te, always_kick(test_df)),
        hand_name: evaluate(y_te, rule_based_go(test_df)),
        tuned_name: evaluate(y_te, rule_based_go(test_df, tuned["max_ydstogo"], tuned["max_yardline_100"])),
    }

    # ---- machine learning: pick the model by grouped CV on training seasons only ----
    models = make_models()
    cv_scores, cv_summary = {}, {}
    cv = GroupKFold(n_splits=5)
    for name, model in models.items():
        s = cross_val_score(model, X_tr, y_tr, groups=train_df["game_id"], cv=cv, scoring="f1")
        cv_summary[name] = {"mean": float(s.mean()), "std": float(s.std())}
        model.fit(X_tr, y_tr)
        prob = model.predict_proba(X_te)[:, 1]
        results[name] = evaluate(y_te, (prob >= 0.5).astype(int), prob)
    best_name = max(cv_summary, key=lambda n: cv_summary[n]["mean"])
    split["best_model"] = best_name
    joblib.dump({"model": models[best_name], "name": best_name, "features": config.FEATURES},
                config.MODEL_DIR / "best_model.joblib")
    for name, model in models.items():
        joblib.dump({"model": model, "name": name, "features": config.FEATURES},
                    config.MODEL_DIR / f"{name.lower().replace(' ', '_')}.joblib")

    rf = models["Random Forest"]
    importances = {f: float(v) for f, v in zip(config.FEATURES, rf.feature_importances_)}

    # ---- backup model: will a go-for-it attempt convert? ----
    go_tr = train_df[(train_df[config.TARGET] == 1)].dropna(subset=["converted"])
    go_te = test_df[(test_df[config.TARGET] == 1)].dropna(subset=["converted"])
    conv_model = RandomForestClassifier(n_estimators=300, min_samples_leaf=10,
                                        random_state=config.RANDOM_STATE, n_jobs=-1)
    conv_model.fit(go_tr[config.FEATURES], go_tr["converted"].astype(int))
    p_conv = conv_model.predict_proba(go_te[config.FEATURES])[:, 1]
    conv = {"test_n": int(len(go_te)), "test_rate": float(go_te["converted"].mean()),
            "roc_auc": float(roc_auc_score(go_te["converted"].astype(int), p_conv)),
            "accuracy": float(accuracy_score(go_te["converted"].astype(int), (p_conv >= 0.5).astype(int)))}
    joblib.dump({"model": conv_model, "name": "Conversion RF", "features": config.FEATURES},
                config.MODEL_DIR / "conversion_model.joblib")

    # ---- save everything ----
    plot_go_rate(df)
    plot_comparison(results)
    metrics = {"counts": counts, "split": {k: (int(v) if isinstance(v, (np.integer,)) else v)
                                           for k, v in split.items()},
               "tuned_rule": tuned, "test_results": results, "cv_f1": cv_summary,
               "conversion_model": conv, "rf_feature_importance": importances}
    (config.RESULTS_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2, default=str))
    write_summary(counts, split, tuned, results, cv_summary, conv, importances)

    print(f"\n=== Test season {test_season} ===")
    for name, r in results.items():
        print(f"{name:40s} acc={fmt(r['accuracy'])} f1={fmt(r['f1'])} "
              f"prec={fmt(r['precision'])} rec={fmt(r['recall'])} auc={fmt(r['roc_auc'])}")
    print(f"\nBest model by CV: {best_name}")
    print("Saved: results/summary.md, results/metrics.json, results/figures/, models/")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile predict.py
"""Enter a 4th-down situation, get a recommendation.

Examples:
    python predict.py                     # runs a built-in demo situation
    python predict.py --yardline 38 --ydstogo 2 --qtr 4 --clock 5:30 --score-diff -3 --home

--yardline is measured from the OPPONENT's end zone: 38 means the opponent's 38-yard line,
70 means your own 30. --score-diff is your score minus the opponent's.
"""
import argparse

import joblib
import pandas as pd

import config
from baseline import rule_based_go


def load_bundle(name: str = "best_model"):
    path = config.MODEL_DIR / f"{name}.joblib"
    if not path.exists():
        raise FileNotFoundError(f"{path.name} not found. Run `python train.py` first.")
    return joblib.load(path)


def game_seconds_remaining(qtr: int, minutes: int, seconds: int) -> int:
    clock = minutes * 60 + seconds
    return (4 - qtr) * 900 + clock if qtr <= 4 else clock


def describe_spot(yardline_100: int) -> str:
    if yardline_100 == 50:
        return "midfield"
    return f"opp {yardline_100}" if yardline_100 < 50 else f"own {100 - yardline_100}"


def recommend(yardline_100, ydstogo, qtr, minutes, seconds, score_differential, is_home,
              decision_bundle=None, conversion_bundle=None):
    if not 1 <= yardline_100 <= 99:
        raise ValueError("yardline must be between 1 and 99 (yards from the opponent's end zone).")
    if not 1 <= ydstogo <= yardline_100:
        raise ValueError("yards to go must be at least 1 and no more than the yardline.")
    if qtr not in (1, 2, 3, 4, 5):
        raise ValueError("quarter must be 1-4 (5 = overtime).")
    if not (0 <= seconds <= 59 and 0 <= minutes <= 15):
        raise ValueError("clock must be between 0:00 and 15:00.")

    decision_bundle = decision_bundle or load_bundle("best_model")
    row = pd.DataFrame([{
        "yardline_100": yardline_100, "ydstogo": ydstogo, "qtr": qtr,
        "game_seconds_remaining": game_seconds_remaining(qtr, minutes, seconds),
        "score_differential": score_differential, "is_home": int(bool(is_home)),
    }])[decision_bundle["features"]]

    p_go = float(decision_bundle["model"].predict_proba(row)[0, 1])
    out = {
        "situation": (f"4th & {ydstogo} at {describe_spot(yardline_100)} | Q{qtr} {minutes}:{seconds:02d} | "
                      f"score diff {score_differential:+d} | {'home' if is_home else 'away'}"),
        "model": decision_bundle["name"],
        "p_go": p_go,
        "recommendation": "Go for it" if p_go >= 0.5 else "Punt / kick",
        "rule_recommendation": "Go for it" if int(rule_based_go(row).iloc[0]) else "Punt / kick",
        "p_convert": None,
    }
    if conversion_bundle is None:
        try:
            conversion_bundle = load_bundle("conversion_model")
        except FileNotFoundError:
            pass
    if conversion_bundle is not None:
        out["p_convert"] = float(conversion_bundle["model"].predict_proba(row[conversion_bundle["features"]])[0, 1])
    return out


def print_result(r):
    print(f"\nSituation: {r['situation']}")
    print(f"Model ({r['model']}) recommends: {r['recommendation'].upper()}   [P(go for it) = {r['p_go']:.2f}]")
    if r["p_convert"] is not None:
        print(f"Estimated chance of converting if they go: {r['p_convert']:.2f}")
    print(f"Simple-rule baseline says: {r['rule_recommendation']}\n")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--yardline", type=int, default=38, help="yards from opponent end zone (default 38)")
    ap.add_argument("--ydstogo", type=int, default=2, help="yards to go (default 2)")
    ap.add_argument("--qtr", type=int, default=4, help="quarter 1-4, 5=OT (default 4)")
    ap.add_argument("--clock", default="5:30", help="clock in the quarter as M:SS (default 5:30)")
    ap.add_argument("--score-diff", type=int, default=-3, help="your score minus opponent (default -3)")
    ap.add_argument("--home", action="store_true", help="offense is the home team")
    a = ap.parse_args()
    m, s = (int(x) for x in a.clock.split(":"))
    try:
        print_result(recommend(a.yardline, a.ydstogo, a.qtr, m, s, a.score_diff, a.home))
    except (ValueError, FileNotFoundError) as err:
        raise SystemExit(f"Error: {err}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile app.py
"""Simple dashboard: enter a 4th-down situation, see the recommendation.

Run with:  streamlit run app.py     (after `python train.py`)
"""
import streamlit as st

import config
from predict import describe_spot, load_bundle, recommend

st.set_page_config(page_title="NFL 4th Down Decision Predictor", page_icon="🏈")
st.title("🏈 NFL Fourth Down Decision Predictor")
st.caption("Trained on historical nflverse play-by-play data. The model learns what coaches "
           "*did* in similar spots, so it reflects real-world (often conservative) habits.")


@st.cache_resource
def get_models():
    try:
        return load_bundle("best_model"), load_bundle("conversion_model")
    except FileNotFoundError:
        return None, None


decision, conversion = get_models()
if decision is None:
    st.error("No trained model found. Run `python train.py` first, then reload this page.")
    st.stop()

left, right = st.columns(2)
with left:
    yardline = st.slider("Yards from opponent end zone", 1, 99, 38,
                         help="38 = opponent's 38-yard line; 70 = your own 30")
    ydstogo = st.slider("Yards to go", 1, 25, 2)
    score_diff = st.number_input("Score difference (your score minus opponent)", -40, 40, -3)
with right:
    qtr = st.selectbox("Quarter", [1, 2, 3, 4, 5], index=3, format_func=lambda q: "OT" if q == 5 else f"Q{q}")
    minutes = st.number_input("Minutes left in quarter", 0, 15, 5)
    seconds = st.number_input("Seconds", 0, 59, 30)
    is_home = st.checkbox("Offense is the home team", value=True)

if st.button("Get recommendation", type="primary"):
    try:
        r = recommend(yardline, ydstogo, qtr, int(minutes), int(seconds), int(score_diff), is_home,
                      decision_bundle=decision, conversion_bundle=conversion)
    except ValueError as err:
        st.error(str(err))
    else:
        st.subheader(f"Recommendation: {r['recommendation']}")
        st.write(r["situation"])
        c1, c2, c3 = st.columns(3)
        c1.metric("P(go for it)", f"{r['p_go']:.0%}")
        if r["p_convert"] is not None:
            c2.metric("Est. conversion chance", f"{r['p_convert']:.0%}")
        c3.metric("Simple-rule baseline", r["rule_recommendation"])


## 2. Install packages
(pandas/numpy/sklearn/matplotlib are already on Colab; this adds pyarrow and streamlit.)

In [ ]:
!pip install -q pyarrow streamlit


## 3. Train and evaluate the models
Downloads the real nflverse seasons (2019–2024 by default), cleans the data, trains Logistic Regression and Random Forest, and compares both against the baselines on the held-out season.

In [ ]:
!python train.py


## 4. View the results summary
Copy this table straight into your report.

In [ ]:
print(open("results/summary.md").read())


## 5. View the comparison figure

In [ ]:
from IPython.display import Image
Image("results/figures/model_comparison.png")


## 6. View the go-rate figure

In [ ]:
from IPython.display import Image
Image("results/figures/go_rate_by_yards_to_go.png")


## 7. Get a prediction for a real situation
Change the numbers below to try other situations.

In [ ]:
!python predict.py --yardline 38 --ydstogo 2 --qtr 4 --clock 5:30 --score-diff -3 --home


## 8. Try a few more situations, including one that might be a weak case
Useful for the "weak case / limitation" part of the report: try close calls (P near 0.5) or situations where you'd expect the model and a human coach to disagree.

In [ ]:
!python predict.py --yardline 60 --ydstogo 1 --qtr 2 --clock 8:00 --score-diff 10
!python predict.py --yardline 45 --ydstogo 4 --qtr 4 --clock 1:30 --score-diff -1 --home


## 9. Look at the raw metrics (for exact numbers / confusion matrix)

In [ ]:
import json
print(json.dumps(json.load(open("results/metrics.json")), indent=2))


## 10. Download everything
Zips the results, models, and processed data so you can save them locally and send the key files back.

In [ ]:
!zip -rq project_outputs.zip results models data/processed
from google.colab import files
files.download("project_outputs.zip")


## Optional: run the dashboard
Colab can't show a Streamlit app inline the normal way. This cell runs it and tunnels it to a public URL (works, but needs the `pyngrok` package). Skip this if you just need numbers for the report — the dashboard isn't required for that.

In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok
import subprocess, time
proc = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(5)
public_url = ngrok.connect(8501)
print(public_url)
